### RAG APplication using Type Sense 

In [2]:
import typesense

In [4]:
client = typesense.Client({
    'nodes': [{
        'host': 'xnvk0d5ta71hroj9p-1.a1.typesense.net', # for typesense cloud use xxx.a1.typesense.net
        'port': '443', # for typesense cloud use 443
        'protocol': 'https' #for typesense cloud use https
    }],
    'api_key': 'jw7HXdp15aLVzraQoVVLqGFX9kVyLcJf',
    'connection_timeout_seconds': 2
})

books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}

print(client.collections.create(books_schema))

{'created_at': 1791145502, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [5]:
with open('books.jsonl' , 'r', encoding= 'utf-8') as f:
    data = f.read()
    client.collections['books'].documents.import_(data)

In [7]:
search_parameters={
    'q':"harry potter",
    'query_by':"title",
    'facet_by'  : "authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 10,
     'highlighted': 'J.K. Rowling',
     'value': 'J.K. Rowling'},
    {'count': 8, 'highlighted': ' Mary GrandPré', 'value': ' Mary GrandPré'},
    {'count': 2, 'highlighted': ' J.K. Rowling', 'value': ' J.K. Rowling'},
    {'count': 1, 'highlighted': 'Bob McCabe', 'value': 'Bob McCabe'},
    {'count': 1, 'highlighted': 'Brian Sibley', 'value': 'Brian Sibley'},
    {'count': 1, 'highlighted': 'David Colbert', 'value': 'David Colbert'},
    {'count': 1, 'highlighted': 'David Baggett', 'value': 'David Baggett'},
    {'count': 1, 'highlighted': 'Melissa Anelli', 'value': 'Melissa Anelli'},
    {'count': 1, 'highlighted': 'John   Williams', 'value': 'John   Williams'},
    {'count': 1,
     'highlighted': ' Shawn E. Klein',
     'value': ' Shawn E. Klein'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 13}}],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating'

### Langchain + Typesense + GroqLLM + RAG Application

In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [15]:
import os
API_KEY = os.getenv("GROQ_API_KEY")

In [16]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
docsearch = Typesense.from_documents(
    docs, embeddings,
    typesense_client_params={
        "host": "xnvk0d5ta71hroj9p-1.a1.typesense.net",  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key":"jw7HXdp15aLVzraQoVVLqGFX9kVyLcJf",
        "typesense_collection_name": "lang-chain"
    },
)

In [18]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

The concepts introduced in "Attention Is All You Need" became the foundation for modern large language models such as BERT, GPT, T5, LLaMA, and many other Transformer-based models used in natural language processing and beyond.

Key concepts from the paper include:
- Self-attention
- Multi-head attention
- Positional encoding
- Encoder-decoder architecture
- Feed-forward neural networks
- Residual connections
- Layer normalization
- Scaled dot-product attention

The scaled dot-product attention is computed using the formula:

Attention(Q, K, V) = softmax((QKáµ€) / âˆšdâ‚–) V

where:
- Q represents the query matrix.
- K represents the key matrix.
- V represents the value matrix.
- dâ‚– is the dimension of the key vectors.

The significance of the paper lies in its simplicity and scalability. Nearly all modern generative AI systems, including ChatGPT and other large language models, are based on the Transformer architecture introduced in this paper.


In [19]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x00000289443ADBB0>, search_kwargs={})

In [20]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content='Summary:\n"Attention Is All You Need" introduced the Transformer architecture, replacing recurrent neural networks with self-attention mechanisms. The model trains faster, captures long-range dependencies more effectively, and serves as the foundation of today\'s large language models.')